# MOBODY Baseline Result Analysis

`/home/MOBODY` 루트에서 실행하는 것을 기준으로, `logs/*.log`와 `runs/` 하위 결과 파일을 자동 탐색해 baseline 결과를 정리합니다.

대상: IQL / DARA / MOBODY, 여러 env, shift/seed별 결과.

In [14]:
from pathlib import Path
import re, json, pickle, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('.')
LOG_DIR = ROOT / 'logs'
RUN_DIR = ROOT / 'runs_friction_shift_sweep_seed1'

print('ROOT:', ROOT.resolve())
print('LOG_DIR exists:', LOG_DIR.exists())
print('RUN_DIR exists:', RUN_DIR.exists())

ROOT: /workspace
LOG_DIR exists: True
RUN_DIR exists: True


## 1. 로그 파일 탐색

In [15]:
def parse_log_filename(path: Path):
    # Example: IQL_walker2d-friction_shift2.0_seed1.log
    stem = path.stem
    m = re.match(r'(?P<policy>[^_]+)_(?P<env>.+)_shift(?P<shift>[-+0-9.]+)_seed(?P<seed>\d+)$', stem)
    if m:
        d = m.groupdict()
        d['shift_level'] = float(d.pop('shift'))
        d['seed'] = int(d['seed'])
        return d
    return {'policy': None, 'env': None, 'shift_level': None, 'seed': None}

log_files = sorted(LOG_DIR.glob('*.log')) if LOG_DIR.exists() else []
print(f'Found {len(log_files)} log files')
for p in log_files[:30]:
    print('-', p)

Found 40 log files
- logs/DARA_halfcheetah-friction_shift0.1_seed1.log
- logs/DARA_halfcheetah-friction_shift0.5_seed1.log
- logs/DARA_halfcheetah-friction_shift2.0_seed1.log
- logs/DARA_halfcheetah-friction_shift5.0_seed1.log
- logs/DARA_hopper-friction_shift0.1_seed1.log
- logs/DARA_hopper-friction_shift0.5_seed1.log
- logs/DARA_hopper-friction_shift2.0_seed1.log
- logs/DARA_hopper-friction_shift5.0_seed1.log
- logs/DARA_walker2d-friction_shift0.1_seed1.log
- logs/DARA_walker2d-friction_shift0.5_seed1.log
- logs/DARA_walker2d-friction_shift2.0_seed1.log
- logs/DARA_walker2d-friction_shift5.0_seed1.log
- logs/IQL_halfcheetah-friction_shift0.1_seed1.log
- logs/IQL_halfcheetah-friction_shift0.5_seed1.log
- logs/IQL_halfcheetah-friction_shift2.0_seed1.log
- logs/IQL_halfcheetah-friction_shift5.0_seed1.log
- logs/IQL_hopper-friction_shift0.1_seed1.log
- logs/IQL_hopper-friction_shift0.5_seed1.log
- logs/IQL_hopper-friction_shift2.0_seed1.log
- logs/IQL_hopper-friction_shift5.0_seed1.log
-

## 2. 로그에서 metric 추출

출력 형식이 다를 수 있어 여러 패턴을 동시에 탐색합니다. 마지막으로 등장한 값을 최종값으로 간주합니다.

In [16]:
METRIC_PATTERNS = {
    'src_eval_return': [
        r'src[_\s-]*eval[_\s-]*return\s*[:=]\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)',
        r'source[_\s-]*eval[_\s-]*return\s*[:=]\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)',
    ],
    'trg_eval_return': [
        r'trg[_\s-]*eval[_\s-]*return\s*[:=]\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)',
        r'target[_\s-]*eval[_\s-]*return\s*[:=]\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)',
    ],
    'eval_return': [
        r'eval[_\s-]*return\s*[:=]\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)',
        r'Evaluation over \d+ episodes\s*[:=]\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)',
        r'Average[_\s-]*Return\s*[:=]\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)',
        r'return\s*[:=]\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)',
    ],
    'normalized_score': [
        r'normalized[_\s-]*score\s*[:=]\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)',
        r'D4RL[_\s-]*score\s*[:=]\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)',
        r'normalized[_\s-]*return\s*[:=]\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)',
    ],
    'loss': [r'loss\s*[:=]\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)'],
}

ERROR_PATTERNS = [
    'Traceback', 'RuntimeError', 'Segmentation fault', 'core dumped', 'Error:',
    'ValueError', 'KeyError', 'FileNotFoundError', 'CUDA out of memory'
]

def extract_last_float(text, patterns):
    vals = []
    for pat in patterns:
        vals.extend(re.findall(pat, text, flags=re.IGNORECASE))
    if not vals:
        return np.nan
    try:
        return float(vals[-1])
    except Exception:
        return np.nan

def parse_log(path: Path):
    meta = parse_log_filename(path)
    text = path.read_text(errors='ignore')
    row = {'log_file': str(path), 'status': 'ok', 'n_lines': text.count('\n') + 1}
    row.update(meta)
    for k, pats in METRIC_PATTERNS.items():
        row[k] = extract_last_float(text, pats)
    errs = [e for e in ERROR_PATTERNS if e.lower() in text.lower()]
    if errs:
        row['status'] = 'error'
        row['error_keywords'] = ', '.join(errs)
        row['tail'] = '\n'.join(text.splitlines()[-25:])
    else:
        row['error_keywords'] = ''
        row['tail'] = '\n'.join(text.splitlines()[-8:])
    return row

log_df = pd.DataFrame([parse_log(p) for p in log_files])
log_df

,log_file,status,n_lines,policy,env,seed,shift_level,src_eval_return,trg_eval_return,eval_return,normalized_score,loss,error_keywords,tail
0,logs/DARA_halfcheetah-friction_shift0.1_seed1.log,error,4080,DARA,halfcheetah-friction,1,0.1,NaN,NaN,NaN,NaN,NaN,Error:,wandb: | 0.195 MB of 0.195 MB uploaded (0.000 ...
1,logs/DARA_halfcheetah-friction_shift0.5_seed1.log,error,4078,DARA,halfcheetah-friction,1,0.5,NaN,NaN,NaN,NaN,NaN,Error:,wandb: | 0.195 MB of 0.195 MB uploaded (0.000 ...
2,logs/DARA_halfcheetah-friction_shift2.0_seed1.log,error,4077,DARA,halfcheetah-friction,1,2.0,NaN,NaN,NaN,NaN,NaN,Error:,wandb: / 0.195 MB of 0.195 MB uploaded (0.000 ...
3,logs/DARA_halfcheetah-friction_shift5.0_seed1.log,error,4080,DARA,halfcheetah-friction,1,5.0,NaN,NaN,NaN,NaN,NaN,Error:,wandb: | 0.195 MB of 0.195 MB uploaded (0.000 ...
4,logs/DARA_hopper-friction_shift0.1_seed1.log,error,4076,DARA,hopper-friction,1,0.1,NaN,NaN,NaN,NaN,NaN,Error:,wandb: / 0.195 MB of 0.195 MB uploaded (0.000 ...
5,logs/DARA_hopper-friction_shift0.5_seed1.log,error,4080,DARA,hopper-friction,1,0.5,NaN,NaN,NaN,NaN,NaN,Error:,wandb: \ 0.195 MB of 0.195 MB uploaded (0.000 ...
6,logs/DARA_hopper-friction_shift2.0_seed1.log,error,4077,DARA,hopper-friction,1,2.0,NaN,NaN,NaN,NaN,NaN,Error:,wandb: - 0.196 MB of 0.196 MB uploaded (0.000 ...
7,logs/DARA_hopper-friction_shift5.0_seed1.log,error,4077,DARA,hopper-friction,1,5.0,NaN,NaN,NaN,NaN,NaN,Error:,wandb: - 0.196 MB of 0.196 MB uploaded (0.000 ...
8,logs/DARA_walker2d-friction_shift0.1_seed1.log,error,4077,DARA,walker2d-friction,1,0.1,NaN,NaN,NaN,NaN,NaN,Error:,wandb: / 0.196 MB of 0.196 MB uploaded (0.000 ...
9,logs/DARA_walker2d-friction_shift0.5_seed1.log,error,4079,DARA,walker2d-friction,1,0.5,NaN,NaN,NaN,NaN,NaN,Error:,wandb: \ 0.195 MB of 0.195 MB uploaded (0.000 ...


In [17]:
if not log_df.empty:
    display_cols = ['policy','env','shift_level','seed','status','src_eval_return','trg_eval_return','eval_return','normalized_score','loss','error_keywords','log_file']
    display(log_df[display_cols].sort_values(['env','shift_level','seed','policy'], na_position='last'))
else:
    print('No log files found.')

,policy,env,shift_level,seed,status,src_eval_return,trg_eval_return,eval_return,normalized_score,loss,error_keywords,log_file
24,MOBODY,PROPOSED_BC_hopper-friction,0.1,1,error,NaN,NaN,NaN,NaN,NaN,Error:,logs/MOBODY_PROPOSED_BC_hopper-friction_shift0...
25,MOBODY,PROPOSED_BC_walker2d-friction,0.1,1,error,NaN,NaN,NaN,NaN,NaN,Error:,logs/MOBODY_PROPOSED_BC_walker2d-friction_shif...
26,MOBODY,PROPOSED_BOTH_hopper-friction,0.1,1,error,NaN,NaN,NaN,NaN,NaN,Error:,logs/MOBODY_PROPOSED_BOTH_hopper-friction_shif...
27,MOBODY,PROPOSED_BOTH_walker2d-friction,0.1,1,error,NaN,NaN,NaN,NaN,NaN,Error:,logs/MOBODY_PROPOSED_BOTH_walker2d-friction_sh...
0,DARA,halfcheetah-friction,0.1,1,error,NaN,NaN,NaN,NaN,NaN,Error:,logs/DARA_halfcheetah-friction_shift0.1_seed1.log
12,IQL,halfcheetah-friction,0.1,1,error,NaN,NaN,NaN,NaN,NaN,Error:,logs/IQL_halfcheetah-friction_shift0.1_seed1.log
28,MOBODY,halfcheetah-friction,0.1,1,error,NaN,NaN,NaN,NaN,1.116222,Error:,logs/MOBODY_halfcheetah-friction_shift0.1_seed...
1,DARA,halfcheetah-friction,0.5,1,error,NaN,NaN,NaN,NaN,NaN,Error:,logs/DARA_halfcheetah-friction_shift0.5_seed1.log
13,IQL,halfcheetah-friction,0.5,1,error,NaN,NaN,NaN,NaN,NaN,Error:,logs/IQL_halfcheetah-friction_shift0.5_seed1.log
29,MOBODY,halfcheetah-friction,0.5,1,error,NaN,NaN,NaN,NaN,0.415784,Error:,logs/MOBODY_halfcheetah-friction_shift0.5_seed...


## 3. 에러 로그 tail 확인

In [18]:
if not log_df.empty:
    err_df = log_df[log_df['status'] == 'error']
    print(f'Error logs: {len(err_df)}')
    for _, r in err_df.iterrows():
        print('=' * 100)
        print(f"{r['policy']} | {r['env']} | shift={r['shift_level']} | seed={r['seed']}")
        print('file:', r['log_file'])
        print('-' * 100)
        print(r['tail'])
else:
    print('No logs.')

Error logs: 40
DARA | halfcheetah-friction | shift=0.1 | seed=1
file: logs/DARA_halfcheetah-friction_shift0.1_seed1.log
----------------------------------------------------------------------------------------------------
wandb: | 0.195 MB of 0.195 MB uploaded (0.000 MB deduped)
wandb: / 0.195 MB of 0.195 MB uploaded (0.000 MB deduped)
wandb: - 0.195 MB of 0.195 MB uploaded (0.000 MB deduped)
wandb: \ 0.195 MB of 0.195 MB uploaded (0.000 MB deduped)
wandb: | 0.195 MB of 0.195 MB uploaded (0.000 MB deduped)
wandb: / 0.195 MB of 0.195 MB uploaded (0.000 MB deduped)
wandb:                                                                                
wandb: 
wandb: Run history:
wandb:                  test/source return ▂▇█▅▇▃▆▅▅▅▄▃▅▆▄▃▃▃▃▄▂▄▃▅▄▃▇▁▄▄▅▃▅▆▅▃▄▄▅▄
wandb:        test/target normalized score ▃▁▂▃▃▄▁▁▄▃▅█▂▄▄▃▅▆▆▅▇▅▄▅▅▄▄▆▄▆▅▃▇▄▃▇▃▄▆▇
wandb:                  test/target return ▃▁▂▃▃▄▁▁▄▃▅█▂▄▄▃▅▆▆▅▇▅▄▅▅▄▄▆▄▆▅▃▇▄▃▇▃▄▆▇
wandb: test/target smooth normalized score ▅▃▁▃▁▃▂▁▃▂▃▆▄▅▄▄▅▆▅▆▅

## 4. `runs/` 결과 파일 자동 탐색

In [19]:
result_files = []
if RUN_DIR.exists():
    for ext in ['*.json', '*.csv', '*.pkl', '*.pickle', '*.npz']:
        result_files.extend(RUN_DIR.rglob(ext))
print(f'Found {len(result_files)} candidate result files')
for p in result_files[:80]:
    print(p)

Found 0 candidate result files


In [20]:
def flatten_dict(d, parent_key='', sep='.'):
    items = []
    for k, v in d.items():
        new_key = f'{parent_key}{sep}{k}' if parent_key else str(k)
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)

KEY_HINTS = ['return', 'score', 'reward', 'eval', 'target', 'source', 'loss', 'normalized']

def summarize_json(path):
    try:
        obj = json.loads(path.read_text(errors='ignore'))
        if isinstance(obj, dict):
            flat = flatten_dict(obj)
            return {k: v for k, v in flat.items() if any(h in k.lower() for h in KEY_HINTS)}
    except Exception:
        return {}
    return {}

json_rows = []
for p in result_files:
    if p.suffix.lower() == '.json':
        s = summarize_json(p)
        if s:
            row = {'file': str(p)}
            row.update(s)
            json_rows.append(row)
json_df = pd.DataFrame(json_rows)
display(json_df.head(50))

""


In [21]:
csv_summaries = []
for p in result_files:
    if p.suffix.lower() == '.csv':
        try:
            df = pd.read_csv(p)
            csv_summaries.append({
                'file': str(p),
                'rows': len(df),
                'cols': list(df.columns),
                'metric_like_cols': [c for c in df.columns if any(h in c.lower() for h in KEY_HINTS)]
            })
        except Exception as e:
            csv_summaries.append({'file': str(p), 'error': str(e)})
csv_summary_df = pd.DataFrame(csv_summaries)
display(csv_summary_df)

""


## 5. Policy별 성능 비교

`trg_eval_return → eval_return → normalized_score → src_eval_return` 순서로 main metric을 선택합니다.

In [22]:
def choose_metric(row):
    for k in ['trg_eval_return', 'eval_return', 'normalized_score', 'src_eval_return']:
        v = row.get(k, np.nan)
        if pd.notna(v):
            return v, k
    return np.nan, None

if not log_df.empty:
    tmp = log_df.copy()
    chosen = tmp.apply(lambda r: choose_metric(r), axis=1)
    tmp['main_metric'] = [x[0] for x in chosen]
    tmp['main_metric_name'] = [x[1] for x in chosen]
    ok_df = tmp[(tmp['status'] == 'ok') & (tmp['main_metric'].notna())].copy()
    display(ok_df[['policy','env','shift_level','seed','main_metric_name','main_metric','log_file']])
else:
    ok_df = pd.DataFrame()
    print('No log dataframe.')

,policy,env,shift_level,seed,main_metric_name,main_metric,log_file


In [23]:
if not ok_df.empty:
    summary = (ok_df.groupby(['env','shift_level','policy'])['main_metric']
               .agg(['count','mean','std','min','max']).reset_index()
               .sort_values(['env','shift_level','mean'], ascending=[True, True, False]))
    display(summary)
else:
    print('No valid metric extracted from logs. 로그 출력 형식을 확인하거나 runs 파일에서 metric을 확인하세요.')

No valid metric extracted from logs. 로그 출력 형식을 확인하거나 runs 파일에서 metric을 확인하세요.


In [24]:
if not ok_df.empty:
    for env_name, sub in ok_df.groupby('env'):
        plt.figure(figsize=(8, 4))
        plot_df = sub.groupby('policy')['main_metric'].mean().sort_values(ascending=False)
        plot_df.plot(kind='bar')
        plt.title(f'Policy comparison: {env_name}')
        plt.ylabel('Main metric')
        plt.xlabel('Policy')
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print('No plot generated.')

No plot generated.


## 6. Env × Policy pivot table

In [25]:
if not ok_df.empty:
    pivot = ok_df.pivot_table(index=['env','shift_level'], columns='policy', values='main_metric', aggfunc='mean')
    display(pivot)
    pivot2 = pivot.copy()
    if 'MOBODY' in pivot2.columns:
        for base in ['IQL', 'DARA']:
            if base in pivot2.columns:
                pivot2[f'MOBODY_minus_{base}'] = pivot2['MOBODY'] - pivot2[base]
                pivot2[f'MOBODY_pct_vs_{base}'] = 100 * (pivot2['MOBODY'] - pivot2[base]) / (pivot2[base].abs() + 1e-8)
    display(pivot2)
else:
    print('No pivot generated.')

No pivot generated.


## 7. 로그에서 학습 곡선 추출 시도

In [26]:
def extract_series_from_log(path, metric_name='eval_return'):
    text = Path(path).read_text(errors='ignore')
    pats = METRIC_PATTERNS.get(metric_name, [])
    vals = []
    for pat in pats:
        vals.extend([float(x) for x in re.findall(pat, text, flags=re.IGNORECASE)])
    return vals

if not log_df.empty:
    for metric_for_curve in ['trg_eval_return', 'eval_return', 'normalized_score']:
        any_plot = False
        for _, r in log_df.iterrows():
            vals = extract_series_from_log(r['log_file'], metric_for_curve)
            if len(vals) >= 2:
                any_plot = True
                plt.figure(figsize=(7, 3))
                plt.plot(vals)
                plt.title(f"{r['policy']} | {r['env']} | {metric_for_curve}")
                plt.xlabel('Logged evaluation index')
                plt.ylabel(metric_for_curve)
                plt.tight_layout()
                plt.show()
        if any_plot:
            break
else:
    print('No logs.')

## 8. 결과 저장

In [27]:
OUT_DIR = ROOT / 'analysis_out'
OUT_DIR.mkdir(exist_ok=True)

if not log_df.empty:
    log_df.to_csv(OUT_DIR / 'parsed_logs.csv', index=False)
    print('Saved:', OUT_DIR / 'parsed_logs.csv')
if 'ok_df' in globals() and not ok_df.empty:
    ok_df.to_csv(OUT_DIR / 'main_metrics_from_logs.csv', index=False)
    print('Saved:', OUT_DIR / 'main_metrics_from_logs.csv')
if 'summary' in globals() and not summary.empty:
    summary.to_csv(OUT_DIR / 'summary_by_env_policy.csv', index=False)
    print('Saved:', OUT_DIR / 'summary_by_env_policy.csv')
if 'json_df' in globals() and not json_df.empty:
    json_df.to_csv(OUT_DIR / 'json_metric_candidates.csv', index=False)
    print('Saved:', OUT_DIR / 'json_metric_candidates.csv')
if 'csv_summary_df' in globals() and not csv_summary_df.empty:
    csv_summary_df.to_csv(OUT_DIR / 'csv_file_summaries.csv', index=False)
    print('Saved:', OUT_DIR / 'csv_file_summaries.csv')

Saved: analysis_out/parsed_logs.csv


## 9. 해석 체크리스트

1. `status == error` 로그가 있는지 확인
2. target/eval return이 정상 추출됐는지 확인
3. env별로 IQL, DARA, MOBODY의 상대 성능 비교
4. 특정 env에서 MOBODY만 불안정하면 로그 tail과 `runs/` 결과 파일 확인
5. 정상 확인 후 seed를 `0,1,2`로 확장